In [1]:
# ============================================================
# GBA Daily Operations Report Generator
# Excel + PDF
# ============================================================

import os
import re
import math
import pandas as pd
import numpy as np

from datetime import datetime, timedelta

# Excel
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# PDF
from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image
)


# ============================================================
# 1. FILE SETTINGS
# ============================================================

# Input Performance Report
INPUT_FILE = "Performance Report 2026.xlsx"

# Excel output
EXCEL_FILE = "Daily_Operations_Report.xlsx"

# PDF output
PDF_FILE = "Daily_Operations_Report.pdf"

# GBA logo
LOGO_FILE = (
    "C:/Users/jason/OneDrive - Greater Bay Airlines Co., Ltd/"
    "Documents/Week 5/header.png"
)

# If None -> automatically use the latest date in the Excel
REPORT_DATE = None


# ============================================================
# 2. DELAY CODE → RESPONSIBLE DEPARTMENT
#    Based on the GBA Delay Codes PDF
# ============================================================

controllable_delay_departments = {

    # Page 1
    "GB": "ISD",
    "GC": "ENG",
    "GD": "GSD",
    "GE": "GSD",
    "GF": "ENG",
    "GL": "GSD",
    "GS": "GSD",
    "GT": "ENG",
    "GU": "GSD",

    "AS": "GSD",

    "CC": "GSD",
    "CD": "GSD",
    "CI": "GSD",
    "CO": "GSD",
    "CP": "GSD",
    "CU": "GSD",
    "CA": "GSD",
    "CE": "GSD",
    "CL": "GSD",

    # Page 2
    "DF": "ENG/FOP",
    "DG": "ENG/GSD",

    "EC": "GSD",
    "ED": "GSD",
    "EF": "OCC",
    "EO": "GSD",

    "FA": "ISD",

    "FB": "FOP",
    "FC": "FOP",
    "FF": "FOP",
    "FL": "ISD",
    "FP": "FOP",
    "FR": "FOP",
    "FS": "FOP",
    "FT": "FOP",

    "OA": "GSD",

    "PB": "GSD",
    "PC": "GSD",
    "PD": "GSD",
    "PE": "GSD",
    "PH": "GSD",
    "PL": "GSD",
    "PO": "GSD",
    "PS": "GSD",
    "PW": "GSD",

    "RC": "FOP",
    "RL": "GSD",
    "RO": "FOP",
    "RS": "FOP",
    "RT": "GSD",

    # Page 3
    "SG": "COM",

    "TA": "ENG",
    "TC": "ENG",
    "TD": "ENG",
    "TL": "ENG",
    "TM": "ENG",
    "TN": "ENG",
    "TS": "ENG",
    "TT": "ENG",
    "TV": "ENG",
}


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def get_delay_code(reason):
    """
    Extract the first two-letter delay code from Reason.

    Examples:
        GF-00:09 LATE REFUELLING  -> GF
        RA-00:15 WEATHER          -> RA
        GB-00:20 ...              -> GB
    """

    if pd.isna(reason):
        return ""

    reason = str(reason).strip().upper()

    match = re.match(r"^([A-Z]{2})", reason)

    if match:
        return match.group(1)

    return ""


def get_responsible_department(delay_code):
    """
    Return Responsible Department for controllable delay codes.

    If the code is not controllable / not found,
    return blank.
    """

    if pd.isna(delay_code):
        return ""

    delay_code = str(delay_code).strip().upper()

    return controllable_delay_departments.get(delay_code, "")


def format_time(value):
    """
    Convert datetime/time value to HH:MM for PDF display.
    """

    if pd.isna(value):
        return "-"

    try:
        return pd.to_datetime(value).strftime("%H:%M")
    except:
        return str(value)


def safe_number(value, decimals=1):
    """
    Safely format numerical values.
    """

    if pd.isna(value):
        return "-"

    try:
        return f"{float(value):.{decimals}f}"
    except:
        return str(value)


# ============================================================
# 4. READ PERFORMANCE REPORT
# ============================================================

print("Reading Performance Report...")

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="2026",
    header=3
)

# Clean column names
df.columns = [str(c).strip() for c in df.columns]

print(f"Total records loaded: {len(df)}")


# ============================================================
# 5. PREPARE DATE COLUMN
# ============================================================

df["Day"] = pd.to_datetime(
    df["Day"],
    errors="coerce"
)

# Remove rows without valid date
df = df[df["Day"].notna()].copy()


# ============================================================
# 6. DETERMINE REPORT DATE
# ============================================================

if REPORT_DATE is None:

    report_date = df["Day"].max().date()

else:

    report_date = pd.to_datetime(
        REPORT_DATE
    ).date()


print(f"Report Date: {report_date}")


# ============================================================
# 7. FILTER DAILY DATA
# ============================================================

daily_df = df[
    df["Day"].dt.date == report_date
].copy()

print(f"Daily records: {len(daily_df)}")


# ============================================================
# 8. CALCULATE DEPARTURE DELAY
# ============================================================

def time_to_minutes(value):
    """
    Convert Excel time / datetime / string / numeric value
    into minutes after midnight.
    """

    if pd.isna(value):
        return np.nan

    # datetime.time
    if hasattr(value, "hour") and hasattr(value, "minute"):
        return (
            value.hour * 60
            + value.minute
            + value.second / 60
        )

    # pandas Timestamp / datetime
    if isinstance(value, (pd.Timestamp, datetime)):
        return (
            value.hour * 60
            + value.minute
            + value.second / 60
        )

    # Numeric Excel time:
    # Excel stores time as fraction of one day
    if isinstance(value, (int, float, np.integer, np.floating)):

        # Example:
        # 18:05 = 18.0833 / 24 = 0.75347
        if 0 <= float(value) < 1:

            return float(value) * 24 * 60

        # If already expressed as minutes
        return float(value)

    # String such as:
    # "18:05"
    # "18:05:00"
    # "18:05:00.000"
    value = str(value).strip()

    try:

        parsed = pd.to_datetime(
            value,
            errors="coerce"
        )

        if pd.isna(parsed):
            return np.nan

        return (
            parsed.hour * 60
            + parsed.minute
            + parsed.second / 60
        )

    except:

        return np.nan


# Convert STD and Out into minutes after midnight
daily_df["STD_minutes"] = daily_df["STD"].apply(
    time_to_minutes
)

daily_df["Out_minutes"] = daily_df["Out"].apply(
    time_to_minutes
)


# Calculate departure delay
daily_df["Delay_minutes"] = (
    daily_df["Out_minutes"]
    - daily_df["STD_minutes"]
)


# ============================================================
# Handle flights crossing midnight
# ============================================================

# Example:
# STD = 23:50
# Out = 00:20
#
# Raw calculation:
# 20 - 1430 = -1410 min
#
# Actual delay:
# +30 min

daily_df.loc[
    daily_df["Delay_minutes"] < -720,
    "Delay_minutes"
] += 1440


# ============================================================
# 9. DELAY CODE & RESPONSIBLE DEPARTMENT
# ============================================================

def get_delay_codes(dep_delay):
    """
    Extract all delay codes from Dep Delay.

    Example:
        SG-00:05,RA-00:24,AG-00:05,AT-00:07
        -> SG, RA, AG, AT
    """

    if pd.isna(dep_delay):
        return []

    dep_delay = str(dep_delay).strip().upper()

    return re.findall(
        r"([A-Z]{2})-",
        dep_delay
    )


def get_responsible_departments(dep_delay):
    """
    Return Responsible Departments for controllable
    delay codes only.
    """

    codes = get_delay_codes(dep_delay)

    departments = []

    for code in codes:

        department = controllable_delay_departments.get(
            code,
            ""
        )

        if department and department not in departments:
            departments.append(department)

    return ", ".join(departments)


# Extract all delay codes from Dep Delay
daily_df["Delay Code"] = daily_df["Dep Delay"].apply(
    lambda x: ", ".join(get_delay_codes(x))
)

# Identify Responsible Department
daily_df["Responsible Department"] = daily_df[
    "Dep Delay"
].apply(
    get_responsible_departments
)


# ============================================================
# 10. BASIC FLIGHT STATISTICS
# ============================================================

# Scheduled Flight Number
scheduled_flights = daily_df[
    daily_df["Flight\nNo"].notna()
]["Flight\nNo"].count()


# Flown Flight Number
flown_flights = (
    daily_df["Status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("flown")
    .sum()
)


# ============================================================
# 11. OTP 30 MINUTES
# ============================================================

delayed_over_30 = (
    daily_df["Delay_minutes"] > 30
).sum()


if scheduled_flights > 0:

    otp_30 = (
        1 -
        delayed_over_30 / scheduled_flights
    )

else:

    otp_30 = np.nan


# OTP target
otp_target = 0.90


# ============================================================
# 12. LOAD FACTOR
# ============================================================

daily_df["Load Factor"] = pd.to_numeric(
    daily_df["Load Factor"],
    errors="coerce"
)

load_factor = daily_df["Load Factor"].mean()


# ============================================================
# 13. TOTAL CARGO
# ============================================================

daily_df["TTL CGO"] = pd.to_numeric(
    daily_df["TTL CGO"],
    errors="coerce"
)

total_cargo = daily_df["TTL CGO"].sum()


# ============================================================
# 14. CONTROLLABLE DELAY COUNT
# ============================================================

controllable_delays = daily_df[
    daily_df["Responsible Department"].astype(str).str.strip() != ""
].copy()

controllable_delay_count = len(
    controllable_delays
)


# ============================================================
# 15. PRINT SUMMARY
# ============================================================

print()
print("=" * 60)
print("GBA DAILY OPERATIONS REPORT")
print("=" * 60)

print(f"Report Date              : {report_date}")
print(f"Scheduled Flight Number  : {scheduled_flights}")
print(f"Flown Flight Number      : {flown_flights}")
print(f"Flights Delayed >30 min  : {delayed_over_30}")

if pd.notna(otp_30):
    print(f"OTP 30 mins              : {otp_30:.1%}")
else:
    print("OTP 30 mins              : -")

if pd.notna(load_factor):
    print(f"Load Factor              : {load_factor:.1%}")
else:
    print("Load Factor              : -")

print(f"Total Cargo              : {total_cargo:,.0f}")
print(f"Controllable Delays      : {controllable_delay_count}")

print("=" * 60)


# ============================================================
# 16. CREATE FLIGHT DETAILS DATAFRAME
# ============================================================

flight_details = daily_df[
    [
        "Flight\nNo",
        "Reg",
        "From",
        "To",
        "STD",
        "Out",
        "Status",
        "Delay_minutes",
        "Reason",
        "Delay Code",
        "Responsible Department",
        "Load Factor",
        "TTL CGO"
    ]
].copy()

#print(f"flight_details是：\n{flight_details}")


flight_details = flight_details.rename(
    columns={
        "Flight\nNo": "Flight No",
        "Delay_minutes": "Departure Delay (min)"
    }
)


# ============================================================
# 17. CREATE DAILY REPORT DATAFRAME
# ============================================================

daily_report = pd.DataFrame({
    "Metric": [
        "Operational Statistics for",
        "Scheduled Flight Number",
        "Flown Flight Number",
        "Flights Delayed >30 min",
        "OTP 30 mins",
        "OTP Target",
        "Load Factor",
        "Total Cargo",
        "Controllable Delays"
    ],

    "Value": [
        str(report_date),
        scheduled_flights,
        flown_flights,
        delayed_over_30,
        otp_30 if pd.notna(otp_30) else "",
        otp_target,
        load_factor if pd.notna(load_factor) else "",
        total_cargo,
        controllable_delay_count
    ]
})


# ============================================================
# 18. EXPORT EXCEL
# ============================================================

print()
print("Creating Excel report...")

wb = Workbook()

# Remove default sheet
ws = wb.active
ws.title = "Daily Report"


# ------------------------------------------------------------
# Daily Report Sheet
# ------------------------------------------------------------

ws["A1"] = "GBA Daily Operations Report"

ws["A1"].font = Font(
    bold=True,
    size=16
)

ws["A3"] = "Report Date"
ws["B3"] = str(report_date)

ws["A4"] = "Scheduled Flight Number"
ws["B4"] = scheduled_flights

ws["A5"] = "Flown Flight Number"
ws["B5"] = flown_flights

ws["A6"] = "Flights Delayed >30 min"
ws["B6"] = delayed_over_30

ws["A7"] = "OTP 30 mins"
ws["B7"] = otp_30 if pd.notna(otp_30) else ""

ws["A8"] = "OTP Target"
ws["B8"] = otp_target

ws["A9"] = "Load Factor"
ws["B9"] = load_factor if pd.notna(load_factor) else ""

ws["A10"] = "Total Cargo"
ws["B10"] = total_cargo

ws["A11"] = "Controllable Delays"
ws["B11"] = controllable_delay_count


# Header style
header_fill = PatternFill(
    "solid",
    fgColor="1F4E78"
)

header_font = Font(
    bold=True,
    color="FFFFFF"
)

thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9")
)


for row in ws.iter_rows(
    min_row=3,
    max_row=11,
    min_col=1,
    max_col=2
):

    for cell in row:
        cell.border = thin_border
        cell.alignment = Alignment(
            vertical="center"
        )


# Number formats
ws["B7"].number_format = "0.0%"
ws["B8"].number_format = "0.0%"
ws["B9"].number_format = "0.0%"
ws["B10"].number_format = "#,##0"


# OTP colour
if pd.notna(otp_30):

    if otp_30 < 0.80:

        ws["B7"].fill = PatternFill(
            "solid",
            fgColor="FFC7CE"
        )

    elif otp_30 < 0.90:

        ws["B7"].fill = PatternFill(
            "solid",
            fgColor="FFF2CC"
        )

    else:

        ws["B7"].fill = PatternFill(
            "solid",
            fgColor="C6EFCE"
        )


ws.column_dimensions["A"].width = 32
ws.column_dimensions["B"].width = 22


# ------------------------------------------------------------
# Flight Details Sheet
# ------------------------------------------------------------

ws2 = wb.create_sheet(
    "Flight Details"
)

# Write headers
for col_idx, column_name in enumerate(
    flight_details.columns,
    start=1
):

    cell = ws2.cell(
        row=1,
        column=col_idx,
        value=column_name
    )

    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
        wrap_text=True
    )

    cell.border = thin_border


# Write data
for row_idx, row in enumerate(
    flight_details.itertuples(index=False),
    start=2
):

    for col_idx, value in enumerate(
        row,
        start=1
    ):

        cell = ws2.cell(
            row=row_idx,
            column=col_idx,
            value=value
        )

        cell.border = thin_border

        cell.alignment = Alignment(
            vertical="center",
            wrap_text=True
        )


# ------------------------------------------------------------
# Excel Formatting
# ------------------------------------------------------------

for row in ws2.iter_rows():

    for cell in row:

        if cell.column in [6]:
            cell.number_format = "0.0"

        if cell.column in [10]:
            cell.number_format = "0.0%"

        if cell.column in [11]:
            cell.number_format = "#,##0"


# Column widths
column_widths = {
    "A": 14,   # Flight No
    "B": 10,   # Reg
    "C": 10,   # STD
    "D": 10,   # Out
    "E": 12,   # Status
    "F": 20,   # Delay
    "G": 42,   # Reason
    "H": 12,   # Delay Code
    "I": 25,   # Responsible Department
    "J": 12,   # Load Factor
    "K": 12    # Cargo
}

for col, width in column_widths.items():

    ws2.column_dimensions[col].width = width


ws2.freeze_panes = "A2"


# ------------------------------------------------------------
# Auto filter
# ------------------------------------------------------------

ws2.auto_filter.ref = ws2.dimensions


# ------------------------------------------------------------
# Save Excel
# ------------------------------------------------------------

wb.save(EXCEL_FILE)

print(f"Excel report created: {EXCEL_FILE}")


# ============================================================
# 19. PDF REPORT
# ============================================================

print()
print("Creating PDF report...")


# ------------------------------------------------------------
# PDF Styles
# ------------------------------------------------------------

styles = getSampleStyleSheet()


title_style = ParagraphStyle(
    "TitleCustom",
    parent=styles["Normal"],
    fontName="Helvetica-Bold",
    fontSize=14,
    leading=16,
    alignment=TA_LEFT
)


subtitle_style = ParagraphStyle(
    "SubtitleCustom",
    parent=styles["Normal"],
    fontName="Helvetica-Bold",
    fontSize=11,
    leading=13,
    alignment=TA_CENTER
)


section_style = ParagraphStyle(
    "SectionCustom",
    parent=styles["Normal"],
    fontName="Helvetica-Bold",
    fontSize=9,
    leading=11,
    alignment=TA_LEFT
)


normal_style = ParagraphStyle(
    "NormalCustom",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=7.5,
    leading=9
)


small_style = ParagraphStyle(
    "SmallCustom",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=6.5,
    leading=8
)


table_header_style = ParagraphStyle(
    "TableHeader",
    parent=styles["Normal"],
    fontName="Helvetica-Bold",
    fontSize=6.5,
    leading=7.5,
    alignment=TA_CENTER
)


table_cell_style = ParagraphStyle(
    "TableCell",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=6.5,
    leading=7.5,
    alignment=TA_CENTER
)


table_left_style = ParagraphStyle(
    "TableLeft",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=6.5,
    leading=7.5,
    alignment=TA_LEFT
)


# ------------------------------------------------------------
# PDF Document
# ------------------------------------------------------------

doc = SimpleDocTemplate(
    PDF_FILE,
    pagesize=A4,
    rightMargin=18 * mm,
    leftMargin=18 * mm,
    topMargin=15 * mm,
    bottomMargin=18 * mm
)


story = []


# ============================================================
# 20. PDF HEADER
# ============================================================

header_left = Paragraph(
    "Daily Operations Report",
    title_style
)

header_middle = Paragraph(
    "Post Operations",
    subtitle_style
)


# Logo
if os.path.exists(LOGO_FILE):

    logo = Image(
        LOGO_FILE,
        width=30 * mm,
        height=12 * mm
    )

else:

    logo = Paragraph(
        "GBA",
        title_style
    )


header_table = Table(
    [
        [
            header_left,
            header_middle,
            logo
        ]
    ],
    colWidths=[
        65 * mm,
        65 * mm,
        35 * mm
    ]
)


header_table.setStyle(
    TableStyle([
        (
            "VALIGN",
            (0, 0),
            (-1, -1),
            "MIDDLE"
        ),

        (
            "ALIGN",
            (0, 0),
            (0, 0),
            "LEFT"
        ),

        (
            "ALIGN",
            (1, 0),
            (1, 0),
            "CENTER"
        ),

        (
            "ALIGN",
            (2, 0),
            (2, 0),
            "RIGHT"
        ),

        (
            "BOTTOMPADDING",
            (0, 0),
            (-1, -1),
            3
        ),

        (
            "TOPPADDING",
            (0, 0),
            (-1, -1),
            0
        )
    ])
)


story.append(header_table)

story.append(
    Spacer(1, 3 * mm)
)


# ============================================================
# 21. REPORT INFORMATION
# ============================================================

info_data = [

    [
        Paragraph(
            "<b>Operational Statistics for</b>",
            normal_style
        ),
        Paragraph(
            f"<b>{report_date}</b>",
            normal_style
        ),
        Paragraph(
            "<b>Issue No.</b>",
            normal_style
        ),
        Paragraph(
            "1",
            normal_style
        )
    ],

    [
        Paragraph(
            "<b>Prepared By</b>",
            normal_style
        ),
        Paragraph(
            "Operations Control Centre",
            normal_style
        ),
        "",
        ""
    ]
]


info_table = Table(
    info_data,
    colWidths=[
        45 * mm,
        55 * mm,
        30 * mm,
        20 * mm
    ]
)


info_table.setStyle(
    TableStyle([
        (
            "GRID",
            (0, 0),
            (-1, -1),
            0.5,
            colors.black
        ),

        (
            "VALIGN",
            (0, 0),
            (-1, -1),
            "MIDDLE"
        ),

        (
            "BACKGROUND",
            (0, 0),
            (0, 0),
            colors.yellow
        ),

        (
            "BACKGROUND",
            (2, 0),
            (2, 0),
            colors.whitesmoke
        ),

        (
            "BACKGROUND",
            (0, 1),
            (0, 1),
            colors.whitesmoke
        ),

        (
            "LEFTPADDING",
            (0, 0),
            (-1, -1),
            4
        ),

        (
            "RIGHTPADDING",
            (0, 0),
            (-1, -1),
            4
        ),

        (
            "TOPPADDING",
            (0, 0),
            (-1, -1),
            3
        ),

        (
            "BOTTOMPADDING",
            (0, 0),
            (-1, -1),
            3
        )
    ])
)


story.append(info_table)

story.append(
    Spacer(1, 5 * mm)
)


# ============================================================
# 22. SECTION 1 – FLIGHT PERFORMANCE
# ============================================================

section_1 = Paragraph(
    "<u>1. Flight Performance</u>",
    section_style
)

story.append(section_1)

story.append(
    Spacer(1, 2 * mm)
)


otp_display = (
    f"{otp_30:.1%}"
    if pd.notna(otp_30)
    else "-"
)

load_factor_display = (
    f"{load_factor:.1%}"
    if pd.notna(load_factor)
    else "-"
)


performance_data = [

    [
        Paragraph(
            "<b>Scheduled<br/>Flights</b>",
            table_header_style
        ),

        Paragraph(
            "<b>Flown<br/>Flights</b>",
            table_header_style
        ),

        Paragraph(
            "<b>Delay<br/>>30 min</b>",
            table_header_style
        ),

        Paragraph(
            "<b>OTP<br/>30 min</b>",
            table_header_style
        ),

        Paragraph(
            "<b>Target</b>",
            table_header_style
        )
    ],

    [
        Paragraph(
            str(scheduled_flights),
            table_cell_style
        ),

        Paragraph(
            str(flown_flights),
            table_cell_style
        ),

        Paragraph(
            str(delayed_over_30),
            table_cell_style
        ),

        Paragraph(
            otp_display,
            table_cell_style
        ),

        Paragraph(
            "90.0%",
            table_cell_style
        )
    ]
]


performance_table = Table(
    performance_data,
    colWidths=[
        30 * mm,
        30 * mm,
        30 * mm,
        30 * mm,
        30 * mm
    ]
)


# OTP background
if pd.notna(otp_30):

    if otp_30 < 0.80:

        otp_bg = colors.HexColor("#FFC7CE")

    elif otp_30 < 0.90:

        otp_bg = colors.HexColor("#FFF2CC")

    else:

        otp_bg = colors.HexColor("#C6EFCE")

else:

    otp_bg = colors.white


performance_table.setStyle(
    TableStyle([
        (
            "GRID",
            (0, 0),
            (-1, -1),
            0.5,
            colors.black
        ),

        (
            "BACKGROUND",
            (0, 0),
            (-1, 0),
            colors.HexColor("#D9EAF7")
        ),

        (
            "BACKGROUND",
            (3, 1),
            (3, 1),
            otp_bg
        ),

        (
            "VALIGN",
            (0, 0),
            (-1, -1),
            "MIDDLE"
        ),

        (
            "ALIGN",
            (0, 0),
            (-1, -1),
            "CENTER"
        ),

        (
            "TOPPADDING",
            (0, 0),
            (-1, -1),
            4
        ),

        (
            "BOTTOMPADDING",
            (0, 0),
            (-1, -1),
            4
        )
    ])
)


story.append(performance_table)

story.append(
    Spacer(1, 5 * mm)
)


# ============================================================
# 23. SECTION 2 – AIRCRAFT / FLIGHT DETAILS
# ============================================================

section_2 = Paragraph(
    "<u>2. Flight Details</u>",
    section_style
)

story.append(section_2)

story.append(
    Spacer(1, 2 * mm)
)


detail_data = [
    [
        Paragraph("<b>Flight</b>", table_header_style),
        Paragraph("<b>Reg</b>", table_header_style),
        Paragraph("<b>From</b>", table_header_style),
        Paragraph("<b>To</b>", table_header_style),
        Paragraph("<b>STD</b>", table_header_style),
        Paragraph("<b>ATD</b>", table_header_style),
        Paragraph("<b>Status</b>", table_header_style),
        Paragraph("<b>Delay<br/>(min)</b>", table_header_style),
        Paragraph("<b>L/F</b>", table_header_style),
        Paragraph("<b>Cargo</b>", table_header_style)
    ]
]


for _, row in daily_df.iterrows():

    flight_no = (
        str(row["Flight\nNo"])
        if pd.notna(row["Flight\nNo"])
        else "-"
    )

    reg = (
        str(row["Reg"])
        if pd.notna(row["Reg"])
        else "-"
    )

    from_airport = (
        str(row["From"])
        if pd.notna(row["From"])
        else "-"
    )

    to_airport = (
        str(row["To"])
        if pd.notna(row["To"])
        else "-"
    )

    std = format_time(
        row["STD"]
    )

    out = format_time(
        row["Out"]
    )

    status = (
        str(row["Status"])
        if pd.notna(row["Status"])
        else "-"
    )

    delay = (
        f"{row['Delay_minutes']:.0f}"
        if pd.notna(row["Delay_minutes"])
        else "-"
    )

    lf = (
        f"{row['Load Factor']:.1%}"
        if pd.notna(row["Load Factor"])
        else "-"
    )

    cargo = (
        f"{row['TTL CGO']:,.0f}"
        if pd.notna(row["TTL CGO"])
        else "-"
    )

    detail_data.append(
        [
            Paragraph(flight_no, table_cell_style),
            Paragraph(reg, table_cell_style),
            Paragraph(from_airport, table_cell_style),
            Paragraph(to_airport, table_cell_style),
            Paragraph(std, table_cell_style),
            Paragraph(out, table_cell_style),
            Paragraph(status, table_cell_style),
            Paragraph(delay, table_cell_style),
            Paragraph(lf, table_cell_style),
            Paragraph(cargo, table_cell_style)
        ]
    )


detail_table = Table(
    detail_data,
    colWidths=[
        20 * mm,
        16 * mm,
        16 * mm,
        16 * mm,
        18 * mm,
        18 * mm,
        22 * mm,
        20 * mm,
        16 * mm,
        18 * mm
    ],
    repeatRows=1
)


detail_table.setStyle(
    TableStyle([
        (
            "GRID",
            (0, 0),
            (-1, -1),
            0.4,
            colors.black
        ),

        (
            "BACKGROUND",
            (0, 0),
            (-1, 0),
            colors.HexColor("#D9EAF7")
        ),

        (
            "VALIGN",
            (0, 0),
            (-1, -1),
            "MIDDLE"
        ),

        (
            "ALIGN",
            (0, 0),
            (-1, -1),
            "CENTER"
        ),

        (
            "TOPPADDING",
            (0, 0),
            (-1, -1),
            2
        ),

        (
            "BOTTOMPADDING",
            (0, 0),
            (-1, -1),
            2
        )
    ])
)


story.append(detail_table)

story.append(
    Spacer(1, 5 * mm)
)


# ============================================================
# 24. SECTION 3 – CONTROLLABLE DELAY RESPONSIBILITY
# ============================================================

section_3 = Paragraph(
    "<u>3. Controllable Delay Responsibility</u>",
    section_style
)

story.append(section_3)

story.append(
    Spacer(1, 2 * mm)
)


# ------------------------------------------------------------
# Only show controllable delays
# ------------------------------------------------------------

controllable_delays = daily_df[
    daily_df["Responsible Department"]
    .astype(str)
    .str.strip()
    != ""
].copy()


delay_responsibility_data = [

    [
        Paragraph("<b>Flight</b>", table_header_style),
        Paragraph("<b>Reason</b>", table_header_style),
        Paragraph("<b>Delay Code</b>", table_header_style),
        Paragraph(
            "<b>Responsible<br/>Department</b>",
            table_header_style
        )
    ]
]


if len(controllable_delays) > 0:

    for _, row in controllable_delays.iterrows():

        flight_no = (
            str(row["Flight\nNo"])
            if pd.notna(row["Flight\nNo"])
            else "-"
        )

        reason = (
            str(row["Reason"])
            if pd.notna(row["Reason"])
            else "-"
        )

        delay_code = (
            str(row["Dep Delay"])
            if pd.notna(row["Dep Delay"])
            and str(row["Dep Delay"]).strip() != ""
            else "-"
        )

        responsible_department = (
            str(row["Responsible Department"])
            if pd.notna(row["Responsible Department"])
            else "-"
        )

        delay_responsibility_data.append(
            [
                Paragraph(
                    flight_no,
                    table_cell_style
                ),

                Paragraph(
                    reason,
                    table_left_style
                ),

                Paragraph(
                    delay_code,
                    table_cell_style
                ),

                Paragraph(
                    responsible_department,
                    table_cell_style
                )
            ]
        )

else:

    delay_responsibility_data.append(
        [
            Paragraph(
                "No controllable delays recorded",
                table_left_style
            ),
            "",
            "",
            ""
        ]
    )


responsibility_table = Table(
    delay_responsibility_data,
    colWidths=[
        25 * mm,
        75 * mm,
        25 * mm,
        40 * mm
    ],
    repeatRows=1
)


responsibility_table.setStyle(
    TableStyle([
        (
            "GRID",
            (0, 0),
            (-1, -1),
            0.4,
            colors.black
        ),

        (
            "BACKGROUND",
            (0, 0),
            (-1, 0),
            colors.HexColor("#D9EAF7")
        ),

        (
            "VALIGN",
            (0, 0),
            (-1, -1),
            "MIDDLE"
        ),

        (
            "ALIGN",
            (0, 0),
            (-1, -1),
            "CENTER"
        ),

        (
            "SPAN",
            (0, 1),
            (-3, 1)
        )
        if len(controllable_delays) == 0
        else (
            "ALIGN",
            (0, 0),
            (-1, -1),
            "CENTER"
        ),

        (
            "TOPPADDING",
            (0, 0),
            (-1, -1),
            3
        ),

        (
            "BOTTOMPADDING",
            (0, 0),
            (-1, -1),
            3
        )
    ])
)


story.append(responsibility_table)

story.append(
    Spacer(1, 5 * mm)
)


# ============================================================
# 25. SECTION 4 – LOAD FACTOR
# ============================================================

section_4 = Paragraph(
    "<u>4. System Load Factor</u>",
    section_style
)

story.append(section_4)

story.append(
    Spacer(1, 2 * mm)
)


load_factor_display = (
    f"{load_factor:.1%}"
    if pd.notna(load_factor)
    else "-"
)


load_data = [

    [
        Paragraph(
            "<b>Daily Average Load Factor</b>",
            table_header_style
        ),

        Paragraph(
            "<b>Total Cargo</b>",
            table_header_style
        )
    ],

    [
        Paragraph(
            load_factor_display,
            table_cell_style
        ),

        Paragraph(
            f"{total_cargo:,.0f}",
            table_cell_style
        )
    ]
]


load_table = Table(
    load_data,
    colWidths=[
        75 * mm,
        75 * mm
    ]
)


load_table.setStyle(
    TableStyle([
        (
            "GRID",
            (0, 0),
            (-1, -1),
            0.5,
            colors.black
        ),

        (
            "BACKGROUND",
            (0, 0),
            (-1, 0),
            colors.HexColor("#D9EAF7")
        ),

        (
            "ALIGN",
            (0, 0),
            (-1, -1),
            "CENTER"
        ),

        (
            "VALIGN",
            (0, 0),
            (-1, -1),
            "MIDDLE"
        ),

        (
            "TOPPADDING",
            (0, 0),
            (-1, -1),
            4
        ),

        (
            "BOTTOMPADDING",
            (0, 0),
            (-1, -1),
            4
        )
    ])
)


story.append(load_table)

story.append(
    Spacer(1, 5 * mm)
)


# ============================================================
# 26. SECTION 5 – MAJOR DEPARTURE DELAY
# ============================================================

section_5 = Paragraph(
    "<u>5. Major Departure Delay (&gt;60 minutes)</u>",
    section_style
)

story.append(section_5)

story.append(
    Spacer(1, 2 * mm)
)


major_delays = daily_df[
    daily_df["Delay_minutes"] > 60
].copy()


major_delay_data = [

    [
        Paragraph("<b>Flight</b>", table_header_style),
        Paragraph("<b>Reg</b>", table_header_style),
        Paragraph("<b>Delay (min)</b>", table_header_style),
        Paragraph("<b>Reason</b>", table_header_style)
    ]
]


if len(major_delays) > 0:

    for _, row in major_delays.iterrows():

        flight_no = (
            str(row["Flight\nNo"])
            if pd.notna(row["Flight\nNo"])
            else "-"
        )

        reg = (
            str(row["Reg"])
            if pd.notna(row["Reg"])
            else "-"
        )

        delay = (
            f"{row['Delay_minutes']:.0f}"
            if pd.notna(row["Delay_minutes"])
            else "-"
        )

        reason = (
            str(row["Reason"])
            if pd.notna(row["Reason"])
            else "-"
        )

        major_delay_data.append(
            [
                Paragraph(
                    flight_no,
                    table_cell_style
                ),

                Paragraph(
                    reg,
                    table_cell_style
                ),

                Paragraph(
                    delay,
                    table_cell_style
                ),

                Paragraph(
                    reason,
                    table_left_style
                )
            ]
        )

else:

    major_delay_data.append(
        [
            Paragraph(
                "No major departure delays recorded",
                table_left_style
            ),
            "",
            "",
            ""
        ]
    )


major_delay_table = Table(
    major_delay_data,
    colWidths=[
        30 * mm,
        25 * mm,
        35 * mm,
        60 * mm
    ],
    repeatRows=1
)


major_delay_table.setStyle(
    TableStyle([
        (
            "GRID",
            (0, 0),
            (-1, -1),
            0.4,
            colors.black
        ),

        (
            "BACKGROUND",
            (0, 0),
            (-1, 0),
            colors.HexColor("#D9EAF7")
        ),

        (
            "VALIGN",
            (0, 0),
            (-1, -1),
            "MIDDLE"
        ),

        (
            "ALIGN",
            (0, 0),
            (-1, -1),
            "CENTER"
        ),

        (
            "TOPPADDING",
            (0, 0),
            (-1, -1),
            3
        ),

        (
            "BOTTOMPADDING",
            (0, 0),
            (-1, -1),
            3
        )
    ])
)


story.append(major_delay_table)


# ============================================================
# 27. PDF FOOTER
# ============================================================

def add_footer(canvas, doc):

    canvas.saveState()

    width, height = A4

    canvas.setFont(
        "Helvetica",
        7
    )

    canvas.drawString(
        18 * mm,
        8 * mm,
        "Confidential"
    )

    canvas.drawRightString(
        width - 18 * mm,
        8 * mm,
        f"Page {doc.page} of 1"
    )

    canvas.restoreState()


# ============================================================
# 28. BUILD PDF
# ============================================================

doc.build(
    story,
    onFirstPage=add_footer,
    onLaterPages=add_footer
)


print(f"PDF report created: {PDF_FILE}")


# ============================================================
# 29. FINAL OUTPUT
# ============================================================

print()
print("=" * 60)
print("REPORT GENERATION COMPLETE")
print("=" * 60)

print(f"Excel : {os.path.abspath(EXCEL_FILE)}")
print(f"PDF   : {os.path.abspath(PDF_FILE)}")

print()
print("Summary:")
print(f"  Report Date             : {report_date}")
print(f"  Scheduled Flights       : {scheduled_flights}")
print(f"  Flown Flights           : {flown_flights}")
print(f"  Delay >30 min           : {delayed_over_30}")

if pd.notna(otp_30):
    print(f"  OTP 30 mins             : {otp_30:.1%}")

if pd.notna(load_factor):
    print(f"  Load Factor             : {load_factor:.1%}")

print(f"  Total Cargo             : {total_cargo:,.0f}")
print(f"  Controllable Delays     : {controllable_delay_count}")

print("=" * 60)

Reading Performance Report...


C:\Users\jason\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


Total records loaded: 5923
Report Date: 2026-08-27
Daily records: 26

GBA DAILY OPERATIONS REPORT
Report Date              : 2026-08-27
Scheduled Flight Number  : 26
Flown Flight Number      : 24
Flights Delayed >30 min  : 10
OTP 30 mins              : 61.5%
Load Factor              : 72.9%
Total Cargo              : 10,809
Controllable Delays      : 3

Creating Excel report...
Excel report created: Daily_Operations_Report.xlsx

Creating PDF report...
PDF report created: Daily_Operations_Report.pdf

REPORT GENERATION COMPLETE
Excel : C:\Users\jason\OneDrive - Greater Bay Airlines Co., Ltd\Documents\OCC Daily Report\Daily_Operations_Report.xlsx
PDF   : C:\Users\jason\OneDrive - Greater Bay Airlines Co., Ltd\Documents\OCC Daily Report\Daily_Operations_Report.pdf

Summary:
  Report Date             : 2026-08-27
  Scheduled Flights       : 26
  Flown Flights           : 24
  Delay >30 min           : 10
  OTP 30 mins             : 61.5%
  Load Factor             : 72.9%
  Total Cargo      